MODUL 1


# M1 — Layered Consolidation Settlement

## 1. Objective

This module calculates primary consolidation settlement of a compressible soil deposit divided into multiple sublayers. The calculation considers the soil stress history through the Overconsolidation Ratio (OCR), compression index (Cc), recompression index (Cr), and initial void ratio (e0).

The settlement is evaluated independently for each layer, and the total settlement is obtained by summing the individual layer settlements.

## 2. Governing Theory

Primary consolidation settlement is estimated using the one-dimensional consolidation theory and the change in void ratio resulting from an increase in effective vertical stress.

The Overconsolidation Ratio is defined as:

\[
OCR = \frac{\sigma'_p}{\sigma'_{v0}}
\]

Where:
- OCR = Overconsolidation Ratio (-)
- σ'p = Preconsolidation pressure (kPa)
- σ'v0 = Initial effective vertical stress (kPa)

Soil classification:
- OCR = 1: Normally Consolidated (NC)
- OCR > 1: Overconsolidated (OC)
- OCR < 1: Underconsolidated (UC)

**Reference:** PennDOT (2025), Publication 293, Chapter 7, Section 7.5.2, p. 7-20.

## 3. Settlement Equations

The settlement calculation distinguishes between the following stress-history conditions.

### 3.1 Normally Consolidated Soil (NC)

For OCR = 1:

\[
S_{c,i} =
\frac{C_{c,i}H_i}{1+e_{0,i}}
\log_{10}\left(
\frac{\sigma'_{vf,i}}{\sigma'_{v0,i}}
\right)
\]

**Reference:**
- PennDOT (2025), Publication 293, Chapter 7, Section 7.5.2.1, p. 7-21.
- Kimmerling (2002), FHWA-SA-02-054, Chapter 5, Section 5.3.3, Equation 5-20, p. 82.

### 3.2 Overconsolidated Soil — Recompression Only

When the final effective stress does not exceed the preconsolidation pressure:

\[
\sigma'_{v0,i} < \sigma'_{vf,i} \leq \sigma'_{p,i}
\]

Settlement is calculated using the recompression index:

\[
S_{c,i} =
\frac{C_{r,i}H_i}{1+e_{0,i}}
\log_{10}\left(
\frac{\sigma'_{vf,i}}{\sigma'_{v0,i}}
\right)
\]

**Reference:**
- PennDOT (2025), Publication 293, Chapter 7, Section 7.5.2.2, p. 7-22, Figure 7.5.2.2-1.
- This equation is derived from the recompression slope (Cr) of the e-log effective stress relationship.

### 3.3 Overconsolidated Soil — Recompression and Virgin Compression

When the final effective stress exceeds the preconsolidation pressure:

\[
\sigma'_{v0,i} < \sigma'_{p,i} < \sigma'_{vf,i}
\]

The settlement consists of two components:

\[
S_{c,i} =
\frac{H_i}{1+e_{0,i}}
\left[
C_{r,i}\log_{10}
\left(\frac{\sigma'_{p,i}}{\sigma'_{v0,i}}\right)
+
C_{c,i}\log_{10}
\left(\frac{\sigma'_{vf,i}}{\sigma'_{p,i}}\right)
\right]
\]

**Reference:**
- PennDOT (2025), Publication 293, Chapter 7, Section 7.5.2.2, p. 7-22.
- Kimmerling (2002), FHWA-SA-02-054, Chapter 5, Section 5.3.3, pp. 83–84.

## 4. Total Settlement

The total primary consolidation settlement is obtained by summing the settlement of all individual sublayers:

\[
S_c = \sum_{i=1}^{n} S_{c,i}
\]

Where:
- Sc = Total primary consolidation settlement (m)
- Sci = Settlement of individual layer i (m)
- n = Number of soil layers (-)

**Reference:** PennDOT (2025), Publication 293, Chapter 7, Sections 7.5.2.1–7.5.2.2, pp. 7-21–7-22.

## 5. Effective Stress Parameters

The initial effective vertical stress is defined as:

\[
\sigma'_{v0} = \sigma_{v0} - u_0
\]

The preconsolidation pressure is calculated from OCR:

\[
\sigma'_p = OCR \cdot \sigma'_{v0}
\]

The final effective vertical stress is:

\[
\sigma'_{vf} = \sigma'_{v0} + \Delta \sigma'_v
\]

Where:
- σv0 = Initial total vertical stress (kPa)
- u0 = Initial pore water pressure (kPa)
- Δσ'v = Effective vertical stress increment (kPa)

**Reference:** PennDOT (2025), Publication 293, Chapter 7, Sections 7.3 and 7.5.2, pp. 7-3–7-8 and 7-20–7-22.

## 6. Implementation Scope and Assumptions

1. The soil deposit is represented by independent horizontal sublayers.
2. Soil parameters are assumed constant within each sublayer.
3. Settlement is calculated using one-dimensional primary consolidation theory.
4. Stress values are evaluated at the center of each sublayer.
5. The effective stress increment is supplied as an input parameter. Its determination depends on the applied loading and stress distribution method.
6. Underconsolidated conditions (OCR < 1) are identified but excluded from the initial NC/OC calculation model.
7. Secondary compression and immediate settlement are excluded.

## 7. References

1. Pennsylvania Department of Transportation (PennDOT). (2025). *Geotechnical Engineering Manual, Publication 293, April 2025 Edition*. Chapter 7 — Settlement, Sections 7.3 and 7.5.2, pp. 7-3–7-8 and 7-20–7-23.  
   https://www.pa.gov/content/dam/copapwp-pagov/en/penndot/documents/public/pubsforms/publications/pub%20293.pdf

2. Kimmerling, R. (2002). *Geotechnical Engineering Circular No. 6: Shallow Foundations*. Report No. FHWA-SA-02-054. Federal Highway Administration, U.S. Department of Transportation. Chapter 5, Section 5.3.3, pp. 81–84.  
   https://rosap.ntl.bts.gov/view/dot/40555


In [1]:

import math


def calculate_layered_settlement(
    layer_thickness,
    ocr,
    cc,
    cr,
    e0,
    sigma_v0_eff,
    delta_sigma_eff
):
    """
    Calculate primary consolidation settlement for multiple soil layers.

    Parameters
    ----------
    layer_thickness : list[float]
        Initial thickness of each soil layer (m).

    ocr : list[float]
        Overconsolidation ratio of each layer (-).
        Only OCR >= 1 is supported in this model.

    cc : list[float]
        Compression index of each layer (-).

    cr : list[float]
        Recompression index of each layer (-).

    e0 : list[float]
        Initial void ratio of each layer (-).

    sigma_v0_eff : list[float]
        Initial effective vertical stress at each layer center (kPa).

    delta_sigma_eff : list[float]
        Effective vertical stress increment at each layer center (kPa).

    Returns
    -------
    dict
        Layer-by-layer calculation results and total settlement.

    Methodology
    -----------
    NC:
        Sc = H*Cc/(1+e0) * log10(sigma_vf/sigma_v0)

    OC - Recompression only:
        Sc = H*Cr/(1+e0) * log10(sigma_vf/sigma_v0)

    OC - Recompression + Virgin compression:
        Sc = H/(1+e0) * (
            Cr*log10(sigma_p/sigma_v0)
            + Cc*log10(sigma_vf/sigma_p)
        )

    References
    ----------
    1. PennDOT (2025), Publication 293,
       Chapter 7, Section 7.5.2.1, p. 7-21.
    2. PennDOT (2025), Publication 293,
       Chapter 7, Section 7.5.2.2, p. 7-22.
    3. Kimmerling (2002), FHWA-SA-02-054,
       Chapter 5, Section 5.3.3,
       Equations 5-20 and 5-22, pp. 82-84.

    Notes
    -----
    - Stress values are evaluated at layer centers.
    - Stress increments must represent effective stress changes.
    - Underconsolidated soil (OCR < 1) is not supported.
    - Immediate and secondary settlement are excluded.
    """

    # --------------------------------------------------
    # 1. INPUT VALIDATION
    # --------------------------------------------------

    inputs = {
        "layer_thickness": layer_thickness,
        "ocr": ocr,
        "cc": cc,
        "cr": cr,
        "e0": e0,
        "sigma_v0_eff": sigma_v0_eff,
        "delta_sigma_eff": delta_sigma_eff
    }

    lengths = {key: len(value) for key, value in inputs.items()}

    if len(set(lengths.values())) != 1:
        raise ValueError(
            f"All input arrays must have equal length. Received: {lengths}"
        )

    n = len(layer_thickness)

    if n == 0:
        raise ValueError("At least one soil layer is required.")

    for name, values in inputs.items():
        for i, value in enumerate(values):
            if not math.isfinite(value):
                raise ValueError(
                    f"{name}[{i}] must be a finite number."
                )

    for i in range(n):

        if layer_thickness[i] <= 0:
            raise ValueError(f"Layer {i+1}: thickness must be > 0.")

        if ocr[i] < 1:
            raise ValueError(
                f"Layer {i+1}: OCR < 1 is outside the supported model."
            )

        if cc[i] <= 0:
            raise ValueError(f"Layer {i+1}: Cc must be > 0.")

        if cr[i] < 0:
            raise ValueError(f"Layer {i+1}: Cr must be >= 0.")

        if e0[i] <= -1:
            raise ValueError(f"Layer {i+1}: e0 must be > -1.")

        if sigma_v0_eff[i] <= 0:
            raise ValueError(
                f"Layer {i+1}: initial effective stress must be > 0."
            )

        if delta_sigma_eff[i] < 0:
            raise ValueError(
                f"Layer {i+1}: stress increment cannot be negative."
            )

    # --------------------------------------------------
    # 2. LAYER-BY-LAYER SETTLEMENT CALCULATION
    # --------------------------------------------------

    layer_results = []

    for i in range(n):

        H = layer_thickness[i]
        OCR = ocr[i]
        Cc = cc[i]
        Cr = cr[i]
        e0_i = e0[i]

        sigma_0 = sigma_v0_eff[i]
        delta_sigma = delta_sigma_eff[i]

        sigma_p = OCR * sigma_0
        sigma_f = sigma_0 + delta_sigma

        # --------------------------------------------------
        # 3. STRESS HISTORY CLASSIFICATION
        # --------------------------------------------------

        if math.isclose(OCR, 1.0, rel_tol=1e-9):
            classification = "NC"
            mechanism = "Virgin compression"

            if sigma_f == sigma_0:
                settlement = 0.0
            else:
                settlement = (
                    H * Cc / (1 + e0_i)
                    * math.log10(sigma_f / sigma_0)
                )

        elif sigma_f <= sigma_p or math.isclose(
            sigma_f, sigma_p, rel_tol=1e-12
        ):
            classification = "OC"
            mechanism = "Recompression only"

            if sigma_f == sigma_0:
                settlement = 0.0
            else:
                settlement = (
                    H * Cr / (1 + e0_i)
                    * math.log10(sigma_f / sigma_0)
                )

        else:
            classification = "OC"
            mechanism = "Recompression + Virgin compression"

            settlement = (
                H / (1 + e0_i)
                * (
                    Cr * math.log10(sigma_p / sigma_0)
                    + Cc * math.log10(sigma_f / sigma_p)
                )
            )

        # --------------------------------------------------
        # 4. STORE LAYER RESULTS
        # --------------------------------------------------

        layer_results.append({
            "layer": i + 1,
            "thickness_m": H,
            "ocr": OCR,
            "classification": classification,
            "mechanism": mechanism,
            "sigma_v0_eff_kpa": sigma_0,
            "sigma_p_kpa": sigma_p,
            "delta_sigma_eff_kpa": delta_sigma,
            "sigma_vf_eff_kpa": sigma_f,
            "settlement_m": settlement,
            "settlement_mm": settlement * 1000
        })

    # --------------------------------------------------
    # 5. TOTAL SETTLEMENT
    # --------------------------------------------------

    total_settlement_m = sum(
        result["settlement_m"] for result in layer_results
    )

    return {
        "number_of_layers": n,
        "layer_results": layer_results,
        "total_settlement_m": total_settlement_m,
        "total_settlement_mm": total_settlement_m * 1000
    }


In [2]:

# TEST 1: Normally Consolidated

result_nc = calculate_layered_settlement(
    layer_thickness=[2.0],
    ocr=[1.0],
    cc=[0.30],
    cr=[0.05],
    e0=[0.80],
    sigma_v0_eff=[100.0],
    delta_sigma_eff=[100.0]
)

expected_nc = (
    2.0 * 0.30 / 1.80 * math.log10(2.0)
)

assert math.isclose(
    result_nc["layer_results"][0]["settlement_m"],
    expected_nc,
    rel_tol=1e-10
)

# TEST 2: OC - Recompression Only

result_oc_recompression = calculate_layered_settlement(
    layer_thickness=[2.0],
    ocr=[2.0],
    cc=[0.30],
    cr=[0.05],
    e0=[0.80],
    sigma_v0_eff=[100.0],
    delta_sigma_eff=[50.0]
)

expected_oc_recompression = (
    2.0 * 0.05 / 1.80 * math.log10(1.5)
)

assert math.isclose(
    result_oc_recompression["layer_results"][0]["settlement_m"],
    expected_oc_recompression,
    rel_tol=1e-10
)

# TEST 3: OC - Recompression + Virgin Compression

result_oc_combined = calculate_layered_settlement(
    layer_thickness=[2.0],
    ocr=[2.0],
    cc=[0.30],
    cr=[0.05],
    e0=[0.80],
    sigma_v0_eff=[100.0],
    delta_sigma_eff=[150.0]
)

expected_oc_combined = (
    2.0 / 1.80
    * (
        0.05 * math.log10(2.0)
        + 0.30 * math.log10(1.25)
    )
)

assert math.isclose(
    result_oc_combined["layer_results"][0]["settlement_m"],
    expected_oc_combined,
    rel_tol=1e-10
)

# TEST 4: Invalid OCR

try:
    calculate_layered_settlement(
        layer_thickness=[2.0],
        ocr=[0.8],
        cc=[0.30],
        cr=[0.05],
        e0=[0.80],
        sigma_v0_eff=[100.0],
        delta_sigma_eff=[50.0]
    )
    raise AssertionError("Expected ValueError for OCR < 1")

except ValueError:
    pass

print("All M1 unit tests passed.")


All M1 unit tests passed.
